<a href="https://colab.research.google.com/github/Andorryu/KU_EECS_836_Final_Project/blob/Task2/Task2/AdvancedCustomTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pip install

In [ ]:
!pip install pyhealth

Authenticate with colab

In [1]:
from google.colab import auth
auth.authenticate_user()

Imports

In [2]:
from google.cloud import bigquery
import tempfile
from pyhealth.datasets import MIMIC3Dataset
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import RNN, RETAIN
from pyhealth.tasks import MortalityPredictionMIMIC3
from pyhealth.trainer import Trainer
from pyhealth.utils import set_seed
from pathlib import Path

BigQuery queries

In [3]:
client = bigquery.Client(project="ml-project-492918")
root = "content/mimiciii"

# Create a single folder
Path("content").mkdir(exist_ok=True)

# Create nested folders (e.g., "parent/child")
Path("content/mimiciii").mkdir(parents=True, exist_ok=True)

W = 24 # number of hours for input window

# Queries
diag_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
"""
proc_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.procedures_icd`
"""
pres_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.prescriptions`
"""
ad_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.admissions`
"""
pat_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.patients`
"""
icu_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.icustays`
"""
lab_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.labevents`
"""
dlab_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.d_labitems`
"""

client.query(diag_query).to_dataframe().to_csv(f"{root}/DIAGNOSES_ICD.csv")
client.query(proc_query).to_dataframe().to_csv(f"{root}/PROCEDURES_ICD.csv")
client.query(pres_query).to_dataframe().to_csv(f"{root}/PRESCRIPTIONS.csv")
client.query(ad_query).to_dataframe().to_csv(f"{root}/ADMISSIONS.csv")
client.query(pat_query).to_dataframe().to_csv(f"{root}/PATIENTS.csv")
client.query(icu_query).to_dataframe().to_csv(f"{root}/ICUSTAYS.csv")
client.query(lab_query).to_dataframe().to_csv(f"{root}/LABEVENTS.csv")
client.query(dlab_query).to_dataframe().to_csv(f"{root}/D_LABITEMS.csv")

Load base dataset

In [4]:
# STEP 1: load data
base_dataset = MIMIC3Dataset(
    root=f"{root}",
    tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
    cache_dir=tempfile.TemporaryDirectory().name,
    dev=True,
)
base_dataset.stats()

No config path provided, using default config


INFO:pyhealth.datasets.mimic3:No config path provided, using default config


Initializing mimic3 dataset from content/mimiciii (dev mode: True)


/usr/local/lib/python3.12/dist-packages/pyhealth/datasets/mimic3.py:59: UserWarning: Events from prescriptions table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(
INFO:pyhealth.datasets.base_dataset:Initializing mimic3 dataset from content/mimiciii (dev mode: True)


Using provided cache_dir: /tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9


...

Dev mode enabled: limiting to 1000 patients


INFO:pyhealth.datasets.base_dataset:Dev mode enabled: limiting to 1000 patients


Caching event dataframe to /tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/global_event_df.parquet...


/usr/local/lib/python3.12/dist-packages/dask/dataframe/core.py:382: UserWarning: Insufficient elements for `head`. 1000 elements requested, only 998 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(
INFO:pyhealth.datasets.base_dataset:Caching event dataframe to /tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/global_event_df.parquet...


Dataset: mimic3
Dev mode: True
Number of patients: 998
Number of events: 769845


Helper Functions

In [5]:
from pyhealth.data import Patient, Event
from datetime import datetime

def getDOB(P: Patient):
  return P.get_events(event_type='patients')[0].attr_dict['dob']

def convertToDT(dtString: str):
  format = '%Y-%m-%d %H:%M:%S'
  dt = datetime.strptime(dtString, format)
  return dt

def getAge(dob: datetime, admissionDate: datetime):
  age = (admissionDate - dob).days // 365
  return age

Custom MIMIC-III BEHRT task

In [9]:

from pyhealth.data import Patient
from datetime import datetime, timedelta
from pyhealth.tasks import BaseTask
import polars as pl
from pyhealth.processors import NestedSequenceProcessor, SequenceProcessor

class BEHRTMortalityPrediction(BaseTask):
  task_name: str = "BEHRTMortalityPrediction"

  """
    codes -> [[labitemID1, labitemID2, ...]] # each labitem is associated with a labevent
    values -> [[value1, value2, ...]] # each value is associated with its corresponding labitemID
    code_types -> ['lab', 'lab', ...]
    ages -> [age, age, age, ...] # one element per labevent, same age throughout (binned per decade)
    time_gaps -> [time_gap1, time_gap2, ...] # time_gap(N) represents time gap between labitemID(N-1) and labitemID(N), (binned per hour)
  """

  input_schema: dict[str, str] = {
      "codes": NestedSequenceProcessor,
      "code_types": SequenceProcessor,
      "values": NestedSequenceProcessor,
      "ages": SequenceProcessor,
      "time_gaps": SequenceProcessor
  }
  output_schema: dict[str, str] = {
      'mortality': 'binary'
  }

  def __call__(self, patient: Patient) -> list[dict[str, any]]:
    input_window_hours = 24 # number of hours after admission time (W)
    samples = []
    dob = convertToDT(getDOB(patient)+" 00:00:00")

    for admission in patient.get_events(event_type="admissions"):

      # skip if patient was discharged during input window
      admission_dischtime = convertToDT(admission.dischtime)
      duration_hour = (
        admission_dischtime - admission.timestamp
      ).total_seconds() / 3600
      if duration_hour <= input_window_hours:
        continue

      # get labevents during input window
      predict_time = admission.timestamp + timedelta(hours=input_window_hours)
      labevents_df = patient.get_events(
        event_type="labevents",
        start=admission.timestamp,
        end=predict_time,
        return_df=True,
      )

      # misc. filtering
      labevents_df = labevents_df.filter(
        pl.col("timestamp").is_not_null()
      )
      labevents_df = labevents_df.with_columns(
        pl.col("labevents/valuenum").cast(pl.Float64, strict=False)
      )
      labevents_df = labevents_df.filter(
        pl.col("labevents/valuenum").is_not_null()
      )

      # skip if admission has invalid data
      if labevents_df.height == 0:
        continue

      # select only timestamp, itemid, and valuenum
      labevents_df = labevents_df.select(
        pl.col("timestamp"),
        pl.col("labevents/itemid"),
        pl.col("labevents/valuenum").cast(pl.Float64)
      )

      # sort
      labevents_df = labevents_df.sort("timestamp")

      # codes
      codes = [labevents_df["labevents/itemid"].to_list()]
      # values
      vals = [labevents_df["labevents/valuenum"].to_list()]
      # code types
      code_types = ["lab" for _ in range(len(codes[0]))]
      # ages
      age = getAge(dob, admission.timestamp)
      ages = [f"age_{age//10}" for _ in range(len(codes[0]))]
      # time gaps
      time_gaps = []
      ts = labevents_df["timestamp"].to_list()
      for i in range(len(ts)):
        if i==0:
          time_gaps.append("gap_none")
          continue
        hrs = (ts[i] - ts[i-1]).total_seconds()//3600
        time_gaps.append(f"gap_{hrs}_{hrs+1}")

      # mortality
      mortality = int(admission.hospital_expire_flag)

      samples.append(
        {
          "patient_id": patient.patient_id,
          "admission_id": admission.hadm_id,
          "codes": codes,
          "code_types": code_types,
          "values": vals,
          "ages": ages,
          "time_gaps": time_gaps,
          "mortality": mortality
        }
      )

    return samples


Set task, split dataset, define model, train, and evaluate

In [10]:
set_seed(42)

# set task
task = BEHRTMortalityPrediction()
sample_dataset = base_dataset.set_task(task)

Setting task BEHRTMortalityPrediction for mimic3 base dataset...


INFO:pyhealth.datasets.base_dataset:Setting task BEHRTMortalityPrediction for mimic3 base dataset...


Task cache paths: task_df=/tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/tasks/BEHRTMortalityPrediction_c7f07ab6-e4c9-5996-a629-ad2d26340a28/task_df.ld, samples=/tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/tasks/BEHRTMortalityPrediction_c7f07ab6-e4c9-5996-a629-ad2d26340a28/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Task cache paths: task_df=/tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/tasks/BEHRTMortalityPrediction_c7f07ab6-e4c9-5996-a629-ad2d26340a28/task_df.ld, samples=/tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/tasks/BEHRTMortalityPrediction_c7f07ab6-e4c9-5996-a629-ad2d26340a28/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Found cached processed samples at /tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/tasks/BEHRTMortalityPrediction_c7f07ab6-e4c9-5996-a629-ad2d26340a28/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.


INFO:pyhealth.datasets.base_dataset:Found cached processed samples at /tmp/tmpbjsdd5yh/ac52524b-dd62-5151-8af1-5d9775f3b6b9/tasks/BEHRTMortalityPrediction_c7f07ab6-e4c9-5996-a629-ad2d26340a28/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
